In [3]:
import openai
import requests
import json
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
URL = "https://nomad-movies.nomadcoders.workers.dev"

In [4]:
def get_popular_movies():
    res = requests.get(f"{URL}/movies")
    data = res.json()
    return [{"id": m["id"], "title": m["title"]} for m in data]

def get_movie_details(id):
    res = requests.get(f"{URL}/movies/{id}")
    return res.json()

def get_movie_credits(id):
    res = requests.get(f"{URL}/movies/{id}/credits")
    return res.json()

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "/movies에서 인기 영화 목록을 가져옵니다.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "/movies/:id에서 특정 ID의 영화 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "/movies/:id/credits에서 출연진 및 제작진을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
]

In [6]:
functions_map = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

def run_agent(user_input):
    messages = [{"role": "user", "content": user_input}]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )

    choice = response.choices[0]

    # tool_call이 없으면 바로 반환
    if choice.finish_reason != "tool_calls":
        return choice.message.content

    # tool_call 실행
    tool_call = choice.message.tool_calls[0]
    fn_name = tool_call.function.name
    fn_args = json.loads(tool_call.function.arguments)

    result = functions_map[fn_name](**fn_args)

    # 결과를 모델에게 전달 → 최종 답변
    messages.append(choice.message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result, ensure_ascii=False),
    })

    final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    return final.choices[0].message.content

In [ ]:
queries = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {run_agent(q)}")
    print("-" * 60)

Q: 지금 인기 있는 영화가 무엇인지 알려줘
A: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Your Heart Will Be Broken**
2. **Avatar: Fire and Ash**
3. **Hoppers**
4. **Crime 101**
5. **Mike & Nick & Nick & Alice**
6. **Shelter**
7. **The Super Mario Galaxy Movie**
8. **Pretty Lethal**
9. **Humint**
10. **GOAT**
11. **Greenland 2: Migration**
12. **War Machine**
13. **The Super Mario Bros. Movie**
14. **Project Hail Mary**
15. **Starbright**
16. **Scream 7**
17. **The Passion of the Christ**
18. **Zootopia 2**
19. **The Shadow's Edge**
20. **Demon Slayer: Kimetsu no Yaiba Infinity Castle**

원하는 영화에 대해 더 알고 싶다면 말씀해 주세요!
------------------------------------------------------------
Q: movie ID 550에 해당하는 영화가 무엇인지 알려줘
